In [1]:
import os
import glob
import polars as pl
import numpy as np
import lightgbm as lgb
from tqdm import tqdm
import gc
import warnings
import json

warnings.filterwarnings("ignore")

os.makedirs("models", exist_ok=True)

# --- CONFIG ---
CONFIG = {
    "raw_data_dir": "./dataset",
    "output_file": "submission_v182_ULTIMATE_SIGNATURES.csv",
    
    "params_m1_t1": {'device': 'cpu', 'objective': 'regression', 'metric': 'rmse', 'boosting_type': 'gbdt', 'learning_rate': 0.0660, 'num_leaves': 127, 'max_depth': 18, 'feature_fraction': 0.8555, 'bagging_fraction': 0.5469, 'bagging_freq': 3, 'lambda_l1': 0.0023, 'lambda_l2': 0.0219, 'min_child_samples': 64, 'n_jobs': -1, 'verbose': -1},
    "params_m1_t2": {'device': 'cpu', 'objective': 'regression', 'metric': 'rmse', 'boosting_type': 'gbdt', 'learning_rate': 0.0716, 'num_leaves': 426, 'max_depth': 20, 'feature_fraction': 0.8225, 'bagging_fraction': 0.6068, 'bagging_freq': 3, 'lambda_l1': 0.0906, 'lambda_l2': 0.0160, 'min_child_samples': 143, 'n_jobs': -1, 'verbose': -1},
    "params_m2_t1": {'device': 'cpu', 'objective': 'regression', 'metric': 'rmse', 'boosting_type': 'gbdt', 'learning_rate': 0.0706, 'num_leaves': 506, 'max_depth': 17, 'feature_fraction': 0.5416, 'bagging_fraction': 0.6436, 'bagging_freq': 3, 'lambda_l1': 0.3465, 'lambda_l2': 0.0155, 'min_child_samples': 91, 'n_jobs': -1, 'verbose': -1},
    "params_m2_t2": {'device': 'cpu', 'objective': 'regression', 'metric': 'rmse', 'boosting_type': 'gbdt', 'learning_rate': 0.0789, 'num_leaves': 511, 'max_depth': 16, 'feature_fraction': 0.7068, 'bagging_fraction': 0.5727, 'bagging_freq': 1, 'lambda_l1': 0.2895, 'lambda_l2': 7.2303, 'min_child_samples': 92, 'n_jobs': -1, 'verbose': -1},
    
    "rounds": 1500,
    "seeds": [42, 888, 2026] # Multi-Seed Bagging
}

def get_safe_files(pattern):
    return [f for f in glob.glob(pattern, recursive=True) if not os.path.basename(f).startswith("._")]

def find_model_path(m_id, split):
    paths = get_safe_files(f"{CONFIG['raw_data_dir']}/**/Model_{m_id}/{split}")
    return paths[0] if paths else None

# ==========================================
# 1. STATIC, EDGES, AND CROSS-COUPLING
# ==========================================
def load_full_static():
    print("   🏗️ Loading Static Physics Properties...")
    static_data = []
    for m_id in [1, 2]:
        base = find_model_path(m_id, "train")
        if not base: continue
        
        f1 = get_safe_files(f"{base}/1d_nodes_static.csv")
        if f1:
            df = pl.read_csv(f1[0]).rename({"node_idx": "node_id"} if "node_idx" in pl.read_csv(f1[0]).columns else {})
            if "surface_elevation" in df.columns: df = df.with_columns((pl.col("surface_elevation") - pl.col("invert_elevation")).alias("calc_depth"))
            if "base_area" not in df.columns: df = df.with_columns(pl.lit(1.0).alias("area"))
            else: df = df.rename({"base_area": "area"})
            cols = ["node_id", "calc_depth", "area", "invert_elevation", "surface_elevation"]
            df = df.select([c for c in cols if c in df.columns]).with_columns([pl.lit(m_id).alias("model_id"), pl.lit(1).alias("node_type")])
            static_data.append(df)
            
        f2 = get_safe_files(f"{base}/2d_nodes_static.csv")
        if f2:
            df = pl.read_csv(f2[0]).rename({"node_idx": "node_id"} if "node_idx" in pl.read_csv(f2[0]).columns else {}).rename({"base_area": "area"} if "base_area" in pl.read_csv(f2[0]).columns else {})
            cols = ["node_id", "area", "elevation", "roughness", "aspect", "curvature", "flow_accumulation", "centroid_elevation", "min_elevation"]
            df = df.select([c for c in cols if c in df.columns]).with_columns([pl.lit(5.0).alias("calc_depth"), pl.lit(m_id).alias("model_id"), pl.lit(2).alias("node_type")])
            static_data.append(df)
            
    return pl.concat(static_data, how="diagonal").fill_null(0.0).with_columns([pl.col("model_id").cast(pl.Int32), pl.col("node_type").cast(pl.Int32), pl.col("node_id").cast(pl.Int32)])

df_static = load_full_static()

def process_edges_and_topology():
    print("   🕸️ Processing Pipes, Channels & Topology...")
    edge_data_list = []
    
    for m_id in [1, 2]:
        base = find_model_path(m_id, "train")
        if not base: continue
        
        f1_stat, f1_idx = get_safe_files(f"{base}/1d_edges_static.csv"), get_safe_files(f"{base}/1d_edge_index.csv")
        if f1_stat and f1_idx:
            edges = pl.read_csv(f1_stat[0])
            idx = pl.read_csv(f1_idx[0])
            if "edge_idx" in edges.columns and "edge_idx" in idx.columns: 
                df_edge = idx.join(edges, on="edge_idx", how="left")
            elif edges.height == idx.height: 
                df_edge = pl.concat([idx, edges], how="horizontal")
            else: 
                df_edge = idx
            
            agg_exprs = [pl.len().alias("pipe_count")]
            if "diameter" in df_edge.columns: agg_exprs.extend([pl.col("diameter").mean().alias("pipe_dia_mean"), pl.col("diameter").max().alias("pipe_dia_max")])
            if "roughness" in df_edge.columns: agg_exprs.append(pl.col("roughness").mean().alias("pipe_rough_mean"))
            if "slope" in df_edge.columns: agg_exprs.append(pl.col("slope").mean().alias("pipe_slope_mean"))
            if "1d_edge_length" in df_edge.columns: agg_exprs.append(pl.col("1d_edge_length").sum().alias("pipe_len_total"))
            if "diameter" in df_edge.columns and "slope" in df_edge.columns and "roughness" in df_edge.columns:
                df_edge = df_edge.with_columns((pl.col("diameter").pow(8/3) * pl.col("slope").abs().pow(0.5) / (pl.col("roughness") + 0.001)).alias("_cap"))
                agg_exprs.append(pl.col("_cap").sum().alias("total_capacity_proxy"))

            from_agg = df_edge.group_by("from_node").agg(agg_exprs).rename({"from_node": "node_id"})
            to_agg = df_edge.group_by("to_node").agg(agg_exprs).rename({"to_node": "node_id"})
            
            cols_to_avg = [c for c in from_agg.columns if c != "node_id"]
            comb = pl.concat([from_agg, to_agg]).group_by("node_id").agg([pl.col(c).mean() for c in cols_to_avg])
            comb = comb.with_columns([pl.col("pipe_count").cast(pl.Float32), pl.lit(m_id).alias("model_id"), pl.lit(1).alias("node_type")])
            edge_data_list.append(comb)

        f2_stat = get_safe_files(f"{base}/2d_edges_static.csv")
        f2_idx = get_safe_files(f"{base}/2d_edge_index.csv")
        f2_nodes = get_safe_files(f"{base}/2d_nodes_static.csv")
        
        if f2_idx and f2_nodes:
            idx = pl.read_csv(f2_idx[0])
            nodes = pl.read_csv(f2_nodes[0]).select(["node_idx", "elevation"]).rename({"node_idx": "node_id"})
            
            degree_counts = pl.concat([idx["from_node"], idx["to_node"]]).value_counts()
            degree = degree_counts.rename({
                degree_counts.columns[0]: "node_id", 
                degree_counts.columns[1]: "degree"
            }).with_columns(pl.col("degree").cast(pl.Float32))
            
            edges_elev = idx.join(nodes, left_on="to_node", right_on="node_id", how="left")
            neighbor_stats = edges_elev.group_by("from_node").agg([
                pl.col("elevation").mean().alias("ne_mean"),
                pl.col("elevation").min().alias("ne_min")
            ]).rename({"from_node": "node_id"})
            
            topo = nodes.join(degree, on="node_id", how="left").fill_null(0).join(neighbor_stats, on="node_id", how="left")
            topo = topo.with_columns([
                (pl.col("elevation") - pl.col("ne_mean").fill_null(pl.col("elevation"))).alias("elev_diff_mean"),
                (pl.col("elevation") - pl.col("ne_min").fill_null(pl.col("elevation"))).alias("elev_diff_min"),
                pl.lit(m_id).alias("model_id"), pl.lit(2).alias("node_type")
            ]).drop("elevation")
            
            if f2_stat:
                edges = pl.read_csv(f2_stat[0])
                if "edge_idx" in edges.columns and "edge_idx" in idx.columns: 
                    df_edge = idx.join(edges, on="edge_idx", how="left")
                elif edges.height == idx.height: 
                    df_edge = pl.concat([idx, edges], how="horizontal")
                else: 
                    df_edge = idx
                
                agg_2d = []
                if "face_length" in df_edge.columns: agg_2d.append(pl.col("face_length").sum().alias("total_face_length"))
                if "2d_length" in df_edge.columns: agg_2d.append(pl.col("2d_length").mean().alias("avg_neighbor_dist"))
                if "slope" in df_edge.columns: agg_2d.extend([pl.col("slope").mean().alias("surf_slope_mean"), pl.col("slope").max().alias("surf_slope_max")])
                
                if agg_2d:
                    from_agg = df_edge.group_by("from_node").agg(agg_2d).rename({"from_node": "node_id"})
                    to_agg = df_edge.group_by("to_node").agg(agg_2d).rename({"to_node": "node_id"})
                    cols_to_avg_2d = [c for c in from_agg.columns if c != "node_id"]
                    comb = pl.concat([from_agg, to_agg]).group_by("node_id").agg([pl.col(c).mean() for c in cols_to_avg_2d])
                    topo = topo.join(comb, on="node_id", how="left")
            
            edge_data_list.append(topo)
            
    if edge_data_list:
        res = pl.concat(edge_data_list, how="diagonal").fill_null(0.0)
        return res.with_columns([pl.col("model_id").cast(pl.Int32), pl.col("node_type").cast(pl.Int32), pl.col("node_id").cast(pl.Int32)])
    return None
df_edges = process_edges_and_topology()

def get_cross_coupling_features():
    print("   🔗 Building Advanced Spatial Cross-Coupling...")
    cross_feats_list = []
    for m_id in [1, 2]:
        base = find_model_path(m_id, "train")
        if not base: continue
        conn_file = get_safe_files(f"{base}/1d2d_connections.csv")
        if not conn_file: continue
        
        conn = pl.read_csv(conn_file[0])
        col_1d = next((c for c in conn.columns if "1d" in c.lower()), None)
        col_2d = next((c for c in conn.columns if "2d" in c.lower()), None)
        if not col_1d or not col_2d: continue
            
        conn = conn.rename({col_1d: "node_1d", col_2d: "node_2d"})
        
        s1 = df_static.filter((pl.col("model_id")==m_id) & (pl.col("node_type")==1))
        if df_edges is not None:
            e1 = df_edges.filter((pl.col("model_id")==m_id) & (pl.col("node_type")==1))
            s1 = s1.join(e1, on=["model_id", "node_type", "node_id"], how="left")
            
        s1_for_2d = s1.select([pl.col("node_id").alias("node_1d"), pl.col("area").alias("coupled_1d_area")])
        if "total_capacity_proxy" in s1.columns:
            s1_for_2d = s1_for_2d.with_columns(s1["total_capacity_proxy"].alias("coupled_1d_capacity"))
            
        map_to_2d = conn.join(s1_for_2d, on="node_1d", how="inner").drop("node_1d").rename({"node_2d": "node_id"}).group_by("node_id").mean()
        map_to_2d = map_to_2d.with_columns([pl.lit(m_id).alias("model_id"), pl.lit(2).alias("node_type")])
        
        s2 = df_static.filter((pl.col("model_id")==m_id) & (pl.col("node_type")==2))
        s2_for_1d = s2.select([pl.col("node_id").alias("node_2d")])
        if "flow_accumulation" in s2.columns:
            s2_for_1d = s2_for_1d.with_columns(s2["flow_accumulation"].alias("coupled_2d_flow_acc"))
            
        map_to_1d = conn.join(s2_for_1d, on="node_2d", how="inner").drop("node_2d").rename({"node_1d": "node_id"}).group_by("node_id").mean()
        map_to_1d = map_to_1d.with_columns([pl.lit(m_id).alias("model_id"), pl.lit(1).alias("node_type")])
        
        cross_feats_list.extend([map_to_2d, map_to_1d])
        
    if cross_feats_list:
        res = pl.concat(cross_feats_list, how="diagonal").fill_null(0.0)
        return res.with_columns([pl.col("model_id").cast(pl.Int32), pl.col("node_type").cast(pl.Int32), pl.col("node_id").cast(pl.Int32)])
    return None

df_cross = get_cross_coupling_features()

# ==========================================
# 2. FULL EXTRACT TIMESERIES 
# ==========================================
def extract_features_timeseries(folder, m_id, e_id, is_train=True):
    f2 = get_safe_files(f"{folder}/*2d_nodes_dynamic*.csv")
    if not f2: return None
    schema = pl.scan_csv(f2[0]).collect_schema()
    rain_col = next((c for c in schema.names() if "rain" in c.lower()), None)
    
    df_rain = pl.read_csv(f2[0], columns=["timestep", rain_col]).with_columns(pl.col("timestep").cast(pl.Int32))
    df_rain = df_rain.group_by("timestep").agg(pl.col(rain_col).mean().alias("rain_step")).sort("timestep")
    peak_ts = df_rain[df_rain["rain_step"].arg_max(), "timestep"]
    
    df_rain = df_rain.with_columns([
        pl.col("timestep").cast(pl.Float32).alias("time_idx"), 
        (pl.col("timestep") - peak_ts).alias("time_from_peak"),
        pl.col("rain_step").ewm_mean(span=4).alias("rain_ewm_1h"), 
        pl.col("rain_step").ewm_mean(span=12).alias("rain_ewm_3h"),
        pl.col("rain_step").ewm_mean(span=24).alias("rain_ewm_6h"), 
        pl.col("rain_step").ewm_mean(span=48).alias("rain_ewm_12h"),
        pl.col("rain_step").ewm_mean(span=96).alias("rain_ewm_24h"), 
        pl.col("rain_step").ewm_mean(span=288).alias("rain_ewm_72h"),
        pl.col("rain_step").cum_sum().alias("rain_cum"),
        pl.col("rain_step").diff().fill_null(0.0).alias("rain_acceleration"),
        pl.col("rain_step").rolling_max(12).fill_null(0.0).alias("rain_rolling_max_3h"),
        (pl.col("rain_step") == 0).cast(pl.Int32).cum_sum().alias("dry_steps_count"),
        pl.col("rain_step").rolling_sum(4).fill_null(0.0).alias("rain_roll_1h"),
    ])
    
    df_rain = df_rain.with_columns([
        pl.col("rain_acceleration").diff().fill_null(0.0).alias("rain_jerk"),
        (pl.col("rain_ewm_1h") - pl.col("rain_ewm_6h")).alias("rain_trend_fast"),
        (pl.col("rain_ewm_6h") - pl.col("rain_ewm_24h")).alias("rain_trend_slow"),
        (pl.col("rain_cum") - pl.col("rain_roll_1h")).alias("antecedent_rain"),
        (pl.col("rain_step") / (df_rain["rain_step"].max() + 0.001)).alias("rain_relative_intensity"),
        pl.col("rain_ewm_1h").shift(-4).fill_null(0.0).alias("rain_future_1h"),
        pl.col("rain_ewm_3h").shift(-12).fill_null(0.0).alias("rain_future_3h"),
    ])

    max_t = df_rain["timestep"].max()
    event_total = df_rain["rain_step"].sum()
    df_rain = df_rain.with_columns([
        (pl.col("timestep")/max_t).alias("time_phase"),
        np.sin(2*np.pi*pl.col("timestep")/max_t).alias("time_sin"),
        np.cos(2*np.pi*pl.col("timestep")/max_t).alias("time_cos"),
        (pl.col("rain_cum")/(event_total+0.001)).alias("rain_cum_pct"),
        pl.lit(event_total).alias("event_total_rain"),
        pl.lit(df_rain["rain_step"].max()).alias("event_max_rain")
    ])

    data_list = []
    for n_type, pattern in [(1, "*1d_nodes_dynamic*.csv"), (2, "*2d_nodes_dynamic*.csv")]:
        f_dyn = get_safe_files(f"{folder}/{pattern}")
        if not f_dyn: continue
        try: df_dyn = pl.read_csv(f_dyn[0])
        except: continue
        if is_train and "water_level" not in df_dyn.columns: continue
        
        df_dyn = df_dyn.with_columns(pl.col("timestep").cast(pl.Int32))
        id_c = "node_idx" if "node_idx" in df_dyn.columns else "node_id"
        
        if "water_level" in df_dyn.columns:
            anchors = df_dyn.filter(pl.col("timestep") < 10).group_by(id_c).agg([
                pl.col("water_level").first().alias("init_wl_first"),
                pl.col("water_level").last().alias("init_wl_last"),
                pl.col("water_level").mean().alias("init_wl_mean"),
                pl.col("water_level").std().fill_null(0.0).alias("init_wl_std"),
                pl.col("water_level").max().alias("init_wl_max"),
                pl.col("water_level").min().alias("init_wl_min"),
                (pl.col("water_level").last() - pl.col("water_level").first()).alias("init_wl_trend"),
                ((pl.col("water_level").last() - pl.col("water_level").first()) / 9.0).alias("init_wl_velocity")
            ])
            df_dyn = df_dyn.join(anchors, on=id_c, how="left")
        else:
            for col in ["init_wl_first", "init_wl_last", "init_wl_mean", "init_wl_std", "init_wl_max", "init_wl_min", "init_wl_trend", "init_wl_velocity"]:
                df_dyn = df_dyn.with_columns(pl.lit(0.0).alias(col))
        
        df_merged = df_dyn.join(df_rain, on="timestep", how="left")
        
        explicit_cols = [
            "time_idx", "time_from_peak", "rain_ewm_1h", "rain_ewm_3h", "rain_ewm_6h", 
            "rain_ewm_12h", "rain_ewm_24h", "rain_ewm_72h", "rain_cum", "rain_acceleration", 
            "rain_rolling_max_3h", "dry_steps_count", "rain_roll_1h", "rain_jerk", 
            "rain_trend_fast", "rain_trend_slow", "antecedent_rain", "rain_relative_intensity", 
            "rain_future_1h", "rain_future_3h", "time_phase", "time_sin", "time_cos", 
            "rain_cum_pct", "event_total_rain", "event_max_rain",
            "init_wl_first", "init_wl_last", "init_wl_mean", "init_wl_std", 
            "init_wl_max", "init_wl_min", "init_wl_trend", "init_wl_velocity"
        ]
        
        selector = [pl.col(id_c).cast(pl.Int32).alias("node_id"), pl.lit(n_type).cast(pl.Int32).alias("node_type"), pl.col("timestep").cast(pl.Int32).alias("timestep_int")] + [pl.col(c).fill_null(0.0) for c in explicit_cols]
        if "water_level" in df_merged.columns: selector.append(pl.col("water_level").alias("target"))
        else: selector.append(pl.col("timestep").cast(pl.Float32).alias("target"))
        
        data_list.append(df_merged.select(selector))
        
    return pl.concat(data_list) if data_list else None

# ==========================================
# 3. BUILD GRAND MATRIX
# ==========================================
print("🚀 Assembling FULL Data Matrix (V182: ULTIMATE SIGNATURES)...")
all_folders = []
for m_id in [1, 2]:
    base = find_model_path(m_id, "train")
    if base:
        for f in get_safe_files(f"{base}/*event_*"):
            try: all_folders.append((m_id, int(os.path.basename(f).split("_")[-1]), f))
            except: pass

train_dfs = []
for m, e, f in tqdm(all_folders, desc="Extracting Train Data"):
    df = extract_features_timeseries(f, m, e, is_train=True)
    if df is not None: train_dfs.append(df.with_columns([pl.lit(m).cast(pl.Int32).alias("model_id"), pl.lit(e).cast(pl.Int32).alias("event_id")]))

df_train = pl.concat(train_dfs)

print("   🧹 Downcasting Float64 to Float32...")
float_cols = [c for c in df_train.columns if df_train[c].dtype == pl.Float64]
if float_cols: df_train = df_train.with_columns([pl.col(c).cast(pl.Float32) for c in float_cols])

# --- NODE SIGNATURES  ---
print("   🧬 Computing Node Signatures...")
node_event = df_train.group_by(["model_id", "node_type", "node_id", "event_id"]).agg([
    pl.col("target").max().alias("ev_max"),
    pl.col("target").min().alias("ev_min"),
    pl.col("target").first().alias("ev_start"),
    pl.col("target").std().alias("ev_std"),
    pl.col("target").mean().alias("ev_mean"),
    pl.col("rain_cum").max().alias("ev_rain_total"),
    pl.col("timestep_int").max().alias("ev_duration")
])

node_event = node_event.with_columns([
    (pl.col("ev_max") - pl.col("ev_start")).alias("amplitude"),
    ((pl.col("ev_max") - pl.col("ev_min")) / (pl.col("ev_rain_total") + 0.001)).alias("sensitivity"),
    (pl.col("ev_std") / (pl.col("ev_mean").abs() + 0.001)).alias("flashiness")
])

signatures = node_event.group_by(["model_id", "node_type", "node_id"]).agg([
    pl.col("sensitivity").median().alias("sig_sensitivity"),
    pl.col("sensitivity").std().fill_null(0.0).alias("sig_sensitivity_var"),
    pl.col("flashiness").median().alias("sig_flashiness"),
    pl.col("amplitude").max().alias("sig_amplitude_max")
])

del node_event; gc.collect()

# Node Stats & Phase Shifts
event_peaks = df_train.group_by(["model_id", "event_id", "node_type", "node_id"]).agg([pl.col("target").max().alias("event_max_wl")])
df_train = df_train.join(event_peaks, on=["model_id", "event_id", "node_type", "node_id"], how="left")
valid_peaks = df_train.filter((pl.col("target") == pl.col("event_max_wl")) & (pl.col("event_max_wl") > pl.col("init_wl_last") + 0.01))
delay_stats = valid_peaks.group_by(["model_id", "event_id", "node_type", "node_id"]).agg([pl.col("time_from_peak").mean().alias("event_peak_delay")]).group_by(["model_id", "node_type", "node_id"]).agg([
    pl.col("event_peak_delay").mean().alias("node_avg_delay"),
    pl.col("event_peak_delay").std().fill_null(0.0).alias("node_std_delay")
])

stats = df_train.group_by(["model_id", "node_type", "node_id"]).agg([
    pl.col("target").mean().alias("node_mean"), pl.col("target").std().fill_null(0.0).alias("node_std"),
    pl.col("target").max().fill_null(0.0).alias("node_max"), pl.col("target").min().fill_null(0.0).alias("node_min")
]).with_columns((pl.col("node_max") - pl.col("node_min")).alias("node_range"))

print("   💾 Saving Node Stats & Signatures for Inference...")
stats.write_csv("models/v182_stats.csv")
signatures.write_csv("models/v182_signatures.csv")
delay_stats.write_csv("models/v182_delay_stats.csv")

# Joins
df_train = df_train.join(stats, on=["model_id", "node_type", "node_id"], how="left")
df_train = df_train.join(signatures, on=["model_id", "node_type", "node_id"], how="left")
df_train = df_train.join(df_static, on=["model_id", "node_type", "node_id"], how="left")
df_train = df_train.join(delay_stats, on=["model_id", "node_type", "node_id"], how="left")
if df_edges is not None: df_train = df_train.join(df_edges, on=["model_id", "node_type", "node_id"], how="left")
if df_cross is not None: df_train = df_train.join(df_cross, on=["model_id", "node_type", "node_id"], how="left")

for c in df_train.columns:
    if c.startswith("init_"): df_train = df_train.with_columns(pl.col(c).fill_null(pl.col("node_mean")))
    elif c.startswith("sig_"): df_train = df_train.with_columns(pl.col(c).fill_null(0.0))
    elif c not in ["target", "event_max_wl", "event_id", "timestep_int", "model_id", "node_type", "node_id"]: df_train = df_train.with_columns(pl.col(c).fill_null(0.0))

print("   🎯 Applying Kaggle Metric Scaling (Sample Weights)...")
group_stds = df_train.group_by(["model_id", "node_type"]).agg([pl.col("target").std().alias("group_std")])
df_train = df_train.join(group_stds, on=["model_id", "node_type"], how="left")

df_train = df_train.with_columns([
    (pl.col("target") - pl.col("node_mean")).alias("residual"),
    (1.0 / (pl.col("group_std").pow(2) + 1e-6)).alias("sample_weight")
])

# 🔥 ULTIMATE PHYSICS + DYNAMIC SIGNATURES 🔥
df_train = df_train.with_columns([
    (pl.col("time_from_peak") - pl.col("node_avg_delay")).alias("relative_time_to_peak"),
    (pl.col("rain_ewm_3h") * pl.col("node_avg_delay")).alias("interact_rain_delay"),
    (pl.col("rain_cum") / (pl.col("node_avg_delay").abs() + 1.0)).alias("interact_cum_delay"),
    
    (pl.col("rain_cum") / (pl.col("calc_depth") * pl.col("area") + 0.1)).alias("proxy_fill_ratio"),
    (pl.col("init_wl_last") - pl.col("node_mean")).alias("interact_anchor_dev"),
    (pl.col("rain_cum") * pl.col("flow_accumulation")).alias("interact_rain_accumulation") if "flow_accumulation" in df_train.columns else pl.lit(0).alias("interact_rain_accumulation"),
    (pl.col("curvature") * pl.col("rain_ewm_3h")).alias("interact_curvature_rain") if "curvature" in df_train.columns else pl.lit(0).alias("interact_curvature_rain"),
    
    # 🧬 DYNAMIC SIGNATURES
    (pl.col("rain_cum") * pl.col("sig_sensitivity")).alias("sig_expected_wl_delta"),
    (pl.col("rain_ewm_1h") * pl.col("sig_sensitivity")).alias("sig_impact_1h"),
    (pl.col("rain_ewm_6h") * pl.col("sig_sensitivity")).alias("sig_impact_6h"),
    (pl.col("sig_amplitude_max") - (pl.col("init_wl_last") - pl.col("node_mean"))).alias("sig_headroom"),
    (pl.col("sig_flashiness") * pl.col("rain_ewm_1h")).alias("sig_flash_risk"),
    (pl.col("sig_sensitivity_var") * pl.col("rain_cum")).alias("sig_uncertainty"),
    
    (pl.col("rain_cum") * pl.col("sig_sensitivity") + pl.col("node_mean") - pl.col("init_wl_last")).alias("sig_expected_vs_actual")
])

if "total_capacity_proxy" in df_train.columns:
    df_train = df_train.with_columns((pl.col("rain_ewm_1h") / (pl.col("total_capacity_proxy") + 0.001)).alias("capacity_utilization"))
if "pipe_dia_max" in df_train.columns:
    df_train = df_train.with_columns((pl.col("rain_cum") / (pl.col("pipe_dia_max").pow(2) * 3.14 / 4 + 0.01)).alias("surcharge_proxy"))

if "coupled_1d_capacity" in df_train.columns:
    df_train = df_train.with_columns([(pl.col("rain_ewm_1h") / (pl.col("coupled_1d_capacity") + 0.1)).alias("cross_choke_risk_1h")])
if "coupled_2d_flow_acc" in df_train.columns:
    df_train = df_train.with_columns([(pl.col("rain_cum") * pl.col("coupled_2d_flow_acc")).alias("cross_inlet_load")])


exclude = ["target", "residual", "sample_weight", "group_std", "event_max_wl", "model_id", "node_type", "node_id", "event_id", "timestep_int"]
all_feats = [c for c in df_train.columns if c not in exclude]

print(f"\n   ⚔️  Training 4 Spatial Models with MULTI-SEED BAGGING ({len(all_feats)} features)...")
models_dict = {}

for m_id in [1, 2]:
    for n_type in [1, 2]:
        name = f"m{m_id}_t{n_type}"
        subset = df_train.filter((pl.col("model_id") == m_id) & (pl.col("node_type") == n_type))
        if subset.height == 0: continue
            
        valid_feats = [f for f in all_feats if subset[f].abs().sum() > 0]
        print(f"   --- {name.upper()} ({len(valid_feats)} valid features) ---")
        
        X = subset.select(valid_feats).to_numpy().astype(np.float32)
        y = subset["residual"].to_numpy().astype(np.float32)
        w = subset["sample_weight"].to_numpy().astype(np.float32)
        
        dtrain = lgb.Dataset(X, label=y, weight=w, free_raw_data=False)
        params = CONFIG[f"params_{name}"]
        
        model_ensemble = []
        for seed in CONFIG["seeds"]:
            params['seed'] = seed
            model = lgb.train(params, dtrain, num_boost_round=CONFIG["rounds"])
            model_ensemble.append(model)
            # 💾
            model_path = f"models/lgbm_v182_m{m_id}_t{n_type}_seed{seed}.txt"
            model.save_model(model_path)
            
        # 💾
        with open(f"models/features_v182_m{m_id}_t{n_type}.json", "w") as f:
            json.dump(valid_feats, f)

        models_dict[(m_id, n_type)] = (model_ensemble, valid_feats)
        
        del X, y, w, dtrain, subset
        gc.collect()

del df_train; gc.collect()

# ==========================================
# 4. INFERENCE
# ==========================================
print("\n   🔮 Fast Inference (Ensemble Averaging)...")
sample_path = get_safe_files(f"{CONFIG['raw_data_dir']}/**/sample_submission.csv")[0]
df_sub_orig = pl.read_csv(sample_path)
test_tasks = df_sub_orig.select(["model_id", "event_id"]).unique().to_dicts()

test_chunks, mapper_chunks = [], []

for task in tqdm(test_tasks, desc="Test Prep"):
    m, e = task['model_id'], task['event_id']
    base = find_model_path(m, "test")
    if not base: continue
    folder = get_safe_files(f"{base}/event_{e}")[0]
    
    df_f = extract_features_timeseries(folder, m, e, is_train=False)
    if df_f is not None: test_chunks.append(df_f.with_columns([pl.lit(m).cast(pl.Int32).alias("model_id"), pl.lit(e).cast(pl.Int32).alias("event_id")]))
        
    sub_slice = df_sub_orig.filter((pl.col("model_id")==m) & (pl.col("event_id")==e))
    for n_type, pattern in [(1, "*1d_nodes_dynamic*.csv"), (2, "*2d_nodes_dynamic*.csv")]:
        files = get_safe_files(f"{folder}/{pattern}")
        if not files: continue
        df_dyn = pl.read_csv(files[0]).with_columns(pl.col("timestep").cast(pl.Int32))
        id_col = "node_idx" if "node_idx" in df_dyn.columns else "node_id"
        targets = df_dyn.filter(pl.col("water_level").is_null()).select([pl.col("timestep").cast(pl.Int32).alias("timestep_int"), pl.col(id_col).cast(pl.Int32).alias("node_id")])
        if targets.height > 0:
            targets = targets.sort(["node_id", "timestep_int"]).with_columns(pl.int_range(0, pl.len()).over("node_id").alias("seq_idx"))
            sub_nodes = sub_slice.filter(pl.col("node_type") == n_type).sort(["node_id", "row_id"]).with_columns(pl.int_range(0, pl.len()).over("node_id").alias("seq_idx"))
            mapped = sub_nodes.join(targets, on=["node_id", "seq_idx"], how="inner")
            mapper_chunks.append(mapped.select(["row_id", "model_id", "event_id", "node_type", "node_id", "timestep_int"]))

df_test = pl.concat(test_chunks)
float_cols = [c for c in df_test.columns if df_test[c].dtype == pl.Float64]
if float_cols: df_test = df_test.with_columns([pl.col(c).cast(pl.Float32) for c in float_cols])

df_test = df_test.join(stats, on=["model_id", "node_type", "node_id"], how="left")
df_test = df_test.join(signatures, on=["model_id", "node_type", "node_id"], how="left")
df_test = df_test.join(df_static, on=["model_id", "node_type", "node_id"], how="left")
df_test = df_test.join(delay_stats, on=["model_id", "node_type", "node_id"], how="left") 
if df_edges is not None: df_test = df_test.join(df_edges, on=["model_id", "node_type", "node_id"], how="left")
if df_cross is not None: df_test = df_test.join(df_cross, on=["model_id", "node_type", "node_id"], how="left")

for c in all_feats:
    if c not in df_test.columns: df_test = df_test.with_columns(pl.lit(0.0).alias(c).cast(pl.Float32))
    if c.startswith("init_"): df_test = df_test.with_columns(pl.col(c).fill_null(pl.col("node_mean")))
    elif c.startswith("sig_"): df_test = df_test.with_columns(pl.col(c).fill_null(0.0))
    else: df_test = df_test.with_columns(pl.col(c).fill_null(0.0))

df_test = df_test.with_columns([
    (pl.col("time_from_peak") - pl.col("node_avg_delay")).alias("relative_time_to_peak"),
    (pl.col("rain_ewm_3h") * pl.col("node_avg_delay")).alias("interact_rain_delay"),
    (pl.col("rain_cum") / (pl.col("node_avg_delay").abs() + 1.0)).alias("interact_cum_delay"),
    
    (pl.col("rain_cum") / (pl.col("calc_depth") * pl.col("area") + 0.1)).alias("proxy_fill_ratio"),
    (pl.col("init_wl_last") - pl.col("node_mean")).alias("interact_anchor_dev"),
    (pl.col("rain_cum") * pl.col("flow_accumulation")).alias("interact_rain_accumulation") if "interact_rain_accumulation" in all_feats else pl.lit(0).alias("interact_rain_accumulation"),
    (pl.col("curvature") * pl.col("rain_ewm_3h")).alias("interact_curvature_rain") if "interact_curvature_rain" in all_feats else pl.lit(0).alias("interact_curvature_rain"),
    
    (pl.col("rain_cum") * pl.col("sig_sensitivity")).alias("sig_expected_wl_delta"),
    (pl.col("rain_ewm_1h") * pl.col("sig_sensitivity")).alias("sig_impact_1h"),
    (pl.col("rain_ewm_6h") * pl.col("sig_sensitivity")).alias("sig_impact_6h"),
    (pl.col("sig_amplitude_max") - (pl.col("init_wl_last") - pl.col("node_mean"))).alias("sig_headroom"),
    (pl.col("sig_flashiness") * pl.col("rain_ewm_1h")).alias("sig_flash_risk"),
    (pl.col("sig_sensitivity_var") * pl.col("rain_cum")).alias("sig_uncertainty"),
    (pl.col("rain_cum") * pl.col("sig_sensitivity") + pl.col("node_mean") - pl.col("init_wl_last")).alias("sig_expected_vs_actual")
])

if "capacity_utilization" in all_feats: df_test = df_test.with_columns((pl.col("rain_ewm_1h") / (pl.col("total_capacity_proxy") + 0.001)).alias("capacity_utilization"))
if "surcharge_proxy" in all_feats: df_test = df_test.with_columns((pl.col("rain_cum") / (pl.col("pipe_dia_max").pow(2) * 3.14 / 4 + 0.01)).alias("surcharge_proxy"))
if "cross_choke_risk_1h" in all_feats: df_test = df_test.with_columns([(pl.col("rain_ewm_1h") / (pl.col("coupled_1d_capacity") + 0.1)).alias("cross_choke_risk_1h")])
if "cross_inlet_load" in all_feats: df_test = df_test.with_columns([(pl.col("rain_cum") * pl.col("coupled_2d_flow_acc")).alias("cross_inlet_load")])

results = []
threads = os.cpu_count() or -1

for m_id in [1, 2]:
    for n_type in [1, 2]:
        subset = df_test.filter((pl.col("model_id") == m_id) & (pl.col("node_type") == n_type))
        if subset.height > 0 and (m_id, n_type) in models_dict:
            model_ensemble, features = models_dict[(m_id, n_type)]
            X_numpy = subset.select(features).to_numpy().astype(np.float32)
            
            preds = np.zeros(X_numpy.shape[0], dtype=np.float32)
            for model in model_ensemble:
                preds += model.predict(X_numpy, num_threads=threads)
            preds /= len(model_ensemble)
            
            results.append(subset.select(["model_id", "event_id", "node_type", "node_id", "timestep_int", "node_mean", "node_min"]).with_columns(pl.Series(preds).alias("pred_residual")))

df_res = pl.concat(results).with_columns(pl.max_horizontal(pl.col("node_mean") + pl.col("pred_residual"), pl.col("node_min")).alias("pred_water_level"))
mapper_df = pl.concat(mapper_chunks)
final_updates = mapper_df.join(df_res.select(["model_id", "event_id", "node_type", "node_id", "timestep_int", "pred_water_level"]), on=["model_id", "event_id", "node_type", "node_id", "timestep_int"], how="left")
df_final = df_sub_orig.join(final_updates.select(["row_id", "pred_water_level"]), on="row_id", how="left").join(stats, on=["model_id", "node_type", "node_id"], how="left")

global_mean = stats["node_mean"].mean()
out = df_final.with_columns(
    pl.col("pred_water_level").fill_null(pl.col("node_mean").fill_null(global_mean))
).select([
    "row_id", "model_id", "event_id", "node_type", "node_id", pl.col("pred_water_level").alias("water_level")
]).sort("row_id")

out.write_csv(CONFIG['output_file'])
print(f"✅ V182 ULTIMATE SIGNATURE ENGINE Saved: {CONFIG['output_file']}")

   🏗️ Loading Static Physics Properties...
   🕸️ Processing Pipes, Channels & Topology...
   🔗 Building Advanced Spatial Cross-Coupling...
🚀 Assembling FULL Data Matrix (V182: ULTIMATE SIGNATURES)...


Extracting Train Data: 100%|██████████| 137/137 [00:39<00:00,  3.46it/s]


   🧹 Downcasting Float64 to Float32...
   🧬 Computing Node Signatures...
   💾 Saving Node Stats & Signatures for Inference...
   🎯 Applying Kaggle Metric Scaling (Sample Weights)...

   ⚔️  Training 4 Spatial Models with MULTI-SEED BAGGING (91 features)...
   --- M1_T1 (73 valid features) ---
   --- M1_T2 (81 valid features) ---
   --- M2_T1 (73 valid features) ---
   --- M2_T2 (81 valid features) ---

   🔮 Fast Inference (Ensemble Averaging)...


Test Prep: 100%|██████████| 59/59 [00:18<00:00,  3.12it/s]


✅ V182 ULTIMATE SIGNATURE ENGINE Saved: submission_v182_ULTIMATE_SIGNATURES.csv


In [2]:
import os
import glob
import polars as pl
import pandas as pd
import numpy as np
import lightgbm as lgb
from scipy.signal import savgol_filter
from tqdm import tqdm
import gc
import warnings
import json

warnings.filterwarnings("ignore")

os.makedirs("models", exist_ok=True)

# --- CONFIG ---
CONFIG = {
    "raw_data_dir": "./dataset",
    "output_file": "submission_v184_PURIFIED_ENGINE.csv",
    
    "params_m1_t1": {'device': 'cpu', 'objective': 'regression', 'metric': 'rmse', 'boosting_type': 'gbdt', 'learning_rate': 0.0660, 'num_leaves': 127, 'max_depth': 18, 'feature_fraction': 0.8555, 'bagging_fraction': 0.5469, 'bagging_freq': 3, 'lambda_l1': 0.0023, 'lambda_l2': 0.0219, 'min_child_samples': 64, 'n_jobs': -1, 'verbose': -1},
    "params_m1_t2": {'device': 'cpu', 'objective': 'regression', 'metric': 'rmse', 'boosting_type': 'gbdt', 'learning_rate': 0.0716, 'num_leaves': 426, 'max_depth': 20, 'feature_fraction': 0.8225, 'bagging_fraction': 0.6068, 'bagging_freq': 3, 'lambda_l1': 0.0906, 'lambda_l2': 0.0160, 'min_child_samples': 143, 'n_jobs': -1, 'verbose': -1},
    "params_m2_t1": {'device': 'cpu', 'objective': 'regression', 'metric': 'rmse', 'boosting_type': 'gbdt', 'learning_rate': 0.0706, 'num_leaves': 506, 'max_depth': 17, 'feature_fraction': 0.5416, 'bagging_fraction': 0.6436, 'bagging_freq': 3, 'lambda_l1': 0.3465, 'lambda_l2': 0.0155, 'min_child_samples': 91, 'n_jobs': -1, 'verbose': -1},
    "params_m2_t2": {'device': 'cpu', 'objective': 'regression', 'metric': 'rmse', 'boosting_type': 'gbdt', 'learning_rate': 0.0789, 'num_leaves': 511, 'max_depth': 16, 'feature_fraction': 0.7068, 'bagging_fraction': 0.5727, 'bagging_freq': 1, 'lambda_l1': 0.2895, 'lambda_l2': 7.2303, 'min_child_samples': 92, 'n_jobs': -1, 'verbose': -1},
    
    "rounds": 1500,
    "seeds": [42, 888, 2026] 
}

def get_safe_files(pattern):
    return [f for f in glob.glob(pattern, recursive=True) if not os.path.basename(f).startswith("._")]

def find_model_path(m_id, split):
    paths = get_safe_files(f"{CONFIG['raw_data_dir']}/**/Model_{m_id}/{split}")
    return paths[0] if paths else None

# ==========================================
# 1. STATIC, EDGES, AND CROSS-COUPLING
# ==========================================
print("🏗️ Loading Static Physics & Undirected Graphs...")
def load_full_static():
    static_data = []
    for m_id in [1, 2]:
        base = find_model_path(m_id, "train")
        if not base: continue
        f1 = get_safe_files(f"{base}/1d_nodes_static.csv")
        if f1:
            df = pl.read_csv(f1[0]).rename({"node_idx": "node_id"} if "node_idx" in pl.read_csv(f1[0]).columns else {})
            if "surface_elevation" in df.columns: df = df.with_columns((pl.col("surface_elevation") - pl.col("invert_elevation")).alias("calc_depth"))
            if "base_area" not in df.columns: df = df.with_columns(pl.lit(1.0).alias("area"))
            else: df = df.rename({"base_area": "area"})
            cols = ["node_id", "calc_depth", "area", "invert_elevation", "surface_elevation"]
            df = df.select([c for c in cols if c in df.columns]).with_columns([pl.lit(m_id).alias("model_id"), pl.lit(1).alias("node_type")])
            static_data.append(df)
            
        f2 = get_safe_files(f"{base}/2d_nodes_static.csv")
        if f2:
            df = pl.read_csv(f2[0]).rename({"node_idx": "node_id"} if "node_idx" in pl.read_csv(f2[0]).columns else {}).rename({"base_area": "area"} if "base_area" in pl.read_csv(f2[0]).columns else {})
            cols = ["node_id", "area", "elevation", "roughness", "aspect", "curvature", "flow_accumulation", "centroid_elevation", "min_elevation"]
            df = df.select([c for c in cols if c in df.columns]).with_columns([pl.lit(5.0).alias("calc_depth"), pl.lit(m_id).alias("model_id"), pl.lit(2).alias("node_type")])
            static_data.append(df)
    return pl.concat(static_data, how="diagonal").fill_null(0.0).with_columns([pl.col("model_id").cast(pl.Int32), pl.col("node_type").cast(pl.Int32), pl.col("node_id").cast(pl.Int32)])

df_static = load_full_static()

def process_edges_and_topology():
    edge_data_list = []
    undirected_edges_list = []
    for m_id in [1, 2]:
        base = find_model_path(m_id, "train")
        if not base: continue
        
        f1_stat, f1_idx = get_safe_files(f"{base}/1d_edges_static.csv"), get_safe_files(f"{base}/1d_edge_index.csv")
        if f1_stat and f1_idx:
            edges, idx = pl.read_csv(f1_stat[0]), pl.read_csv(f1_idx[0])
            df_edge = idx.join(edges, on="edge_idx", how="left") if "edge_idx" in edges.columns else pl.concat([idx, edges], how="horizontal") if edges.height == idx.height else idx
            
            undirected_1d = pl.concat([idx.select(["from_node", "to_node"]), idx.select([pl.col("to_node").alias("from_node"), pl.col("from_node").alias("to_node")])]).unique().with_columns([pl.lit(m_id).alias("model_id"), pl.lit(1).alias("node_type")])
            undirected_edges_list.append(undirected_1d)

            agg_exprs = [pl.len().alias("pipe_count"), pl.col("diameter").mean().alias("pipe_dia_mean"), pl.col("diameter").max().alias("pipe_dia_max")]
            if "diameter" in df_edge.columns and "slope" in df_edge.columns:
                df_edge = df_edge.with_columns((pl.col("diameter").pow(8/3) * pl.col("slope").abs().pow(0.5) / (pl.col("roughness") + 0.001)).alias("_cap"))
                agg_exprs.append(pl.col("_cap").sum().alias("total_capacity_proxy"))
            comb = pl.concat([df_edge.group_by("from_node").agg(agg_exprs).rename({"from_node": "node_id"}), df_edge.group_by("to_node").agg(agg_exprs).rename({"to_node": "node_id"})])
            cols_to_avg = [c for c in comb.columns if c != "node_id"]
            comb = comb.group_by("node_id").agg([pl.col(c).mean() for c in cols_to_avg]).with_columns([pl.col("pipe_count").cast(pl.Float32), pl.lit(m_id).alias("model_id"), pl.lit(1).alias("node_type")])
            edge_data_list.append(comb)

        f2_stat, f2_idx, f2_nodes = get_safe_files(f"{base}/2d_edges_static.csv"), get_safe_files(f"{base}/2d_edge_index.csv"), get_safe_files(f"{base}/2d_nodes_static.csv")
        if f2_idx and f2_nodes:
            idx, nodes = pl.read_csv(f2_idx[0]), pl.read_csv(f2_nodes[0]).rename({"node_idx": "node_id"})
            
            undirected_2d = pl.concat([idx.select(["from_node", "to_node"]), idx.select([pl.col("to_node").alias("from_node"), pl.col("from_node").alias("to_node")])]).unique().with_columns([pl.lit(m_id).alias("model_id"), pl.lit(2).alias("node_type")])
            undirected_edges_list.append(undirected_2d)

            degree_counts = pl.concat([idx["from_node"], idx["to_node"]]).value_counts()
            degree = degree_counts.rename({degree_counts.columns[0]: "node_id", degree_counts.columns[1]: "degree"}).with_columns(pl.col("degree").cast(pl.Float32))
            edges_elev = idx.join(nodes, left_on="to_node", right_on="node_id", how="left")
            neighbor_stats = edges_elev.group_by("from_node").agg([pl.col("elevation").mean().alias("ne_mean"), pl.col("elevation").min().alias("ne_min")]).rename({"from_node": "node_id"})
            topo = nodes.join(degree, on="node_id", how="left").fill_null(0).join(neighbor_stats, on="node_id", how="left")
            topo = topo.with_columns([(pl.col("elevation") - pl.col("ne_mean").fill_null(pl.col("elevation"))).alias("elev_diff_mean"), (pl.col("elevation") - pl.col("ne_min").fill_null(pl.col("elevation"))).alias("elev_diff_min"), pl.lit(m_id).alias("model_id"), pl.lit(2).alias("node_type")]).drop("elevation")
            if f2_stat:
                edges = pl.read_csv(f2_stat[0])
                df_edge = idx.join(edges, on="edge_idx", how="left") if "edge_idx" in edges.columns else pl.concat([idx, edges], how="horizontal") if edges.height == idx.height else idx
                agg_2d = []
                if "face_length" in df_edge.columns: agg_2d.append(pl.col("face_length").sum().alias("total_face_length"))
                if "2d_length" in df_edge.columns: agg_2d.append(pl.col("2d_length").mean().alias("avg_neighbor_dist"))
                if "slope" in df_edge.columns: agg_2d.extend([pl.col("slope").mean().alias("surf_slope_mean"), pl.col("slope").max().alias("surf_slope_max")])
                if agg_2d:
                    from_agg = df_edge.group_by("from_node").agg(agg_2d).rename({"from_node": "node_id"})
                    to_agg = df_edge.group_by("to_node").agg(agg_2d).rename({"to_node": "node_id"})
                    cols_to_avg_2d = [c for c in from_agg.columns if c != "node_id"]
                    comb = pl.concat([from_agg, to_agg]).group_by("node_id").agg([pl.col(c).mean() for c in cols_to_avg_2d])
                    topo = topo.join(comb, on="node_id", how="left")
            edge_data_list.append(topo)
            
    res_edges = pl.concat(edge_data_list, how="diagonal").fill_null(0.0).with_columns([pl.col("model_id").cast(pl.Int32), pl.col("node_type").cast(pl.Int32), pl.col("node_id").cast(pl.Int32)]) if edge_data_list else None
    res_undirected = pl.concat(undirected_edges_list).with_columns([pl.col("model_id").cast(pl.Int32), pl.col("node_type").cast(pl.Int32)]) if undirected_edges_list else None
    return res_edges, res_undirected

df_edges, df_undirected_graph = process_edges_and_topology()

def get_cross_coupling_features():
    cross_feats_list = []
    for m_id in [1, 2]:
        base = find_model_path(m_id, "train")
        if not base: continue
        conn_file = get_safe_files(f"{base}/1d2d_connections.csv")
        if not conn_file: continue
        conn = pl.read_csv(conn_file[0])
        col_1d, col_2d = next((c for c in conn.columns if "1d" in c.lower()), None), next((c for c in conn.columns if "2d" in c.lower()), None)
        if not col_1d or not col_2d: continue
        conn = conn.rename({col_1d: "node_1d", col_2d: "node_2d"})
        s1 = df_static.filter((pl.col("model_id")==m_id) & (pl.col("node_type")==1))
        if df_edges is not None: s1 = s1.join(df_edges.filter((pl.col("model_id")==m_id) & (pl.col("node_type")==1)), on=["model_id", "node_type", "node_id"], how="left")
        s1_for_2d = s1.select([pl.col("node_id").alias("node_1d"), pl.col("area").alias("coupled_1d_area")])
        if "total_capacity_proxy" in s1.columns: s1_for_2d = s1_for_2d.with_columns(s1["total_capacity_proxy"].alias("coupled_1d_capacity"))
        map_to_2d = conn.join(s1_for_2d, on="node_1d", how="inner").drop("node_1d").rename({"node_2d": "node_id"}).group_by("node_id").mean().with_columns([pl.lit(m_id).alias("model_id"), pl.lit(2).alias("node_type")])
        s2 = df_static.filter((pl.col("model_id")==m_id) & (pl.col("node_type")==2))
        s2_for_1d = s2.select([pl.col("node_id").alias("node_2d")])
        if "flow_accumulation" in s2.columns: s2_for_1d = s2_for_1d.with_columns(s2["flow_accumulation"].alias("coupled_2d_flow_acc"))
        map_to_1d = conn.join(s2_for_1d, on="node_2d", how="inner").drop("node_2d").rename({"node_1d": "node_id"}).group_by("node_id").mean().with_columns([pl.lit(m_id).alias("model_id"), pl.lit(1).alias("node_type")])
        cross_feats_list.extend([map_to_2d, map_to_1d])
    return pl.concat(cross_feats_list, how="diagonal").fill_null(0.0).with_columns([pl.col("model_id").cast(pl.Int32), pl.col("node_type").cast(pl.Int32), pl.col("node_id").cast(pl.Int32)]) if cross_feats_list else None

df_cross = get_cross_coupling_features()

# ==========================================
# 2. FULL EXTRACT TIMESERIES
# ==========================================
def extract_features_timeseries(folder, m_id, e_id, is_train=True):
    f2 = get_safe_files(f"{folder}/*2d_nodes_dynamic*.csv")
    if not f2: return None
    schema = pl.scan_csv(f2[0]).collect_schema()
    rain_col = next((c for c in schema.names() if "rain" in c.lower()), None)
    
    df_rain = pl.read_csv(f2[0], columns=["timestep", rain_col]).with_columns(pl.col("timestep").cast(pl.Int32))
    df_rain = df_rain.group_by("timestep").agg(pl.col(rain_col).mean().alias("rain_step")).sort("timestep")
    peak_ts = df_rain[df_rain["rain_step"].arg_max(), "timestep"]
    
    df_rain = df_rain.with_columns([
        pl.col("timestep").cast(pl.Float32).alias("time_idx"), 
        (pl.col("timestep") - peak_ts).alias("time_from_peak"),
        pl.col("rain_step").ewm_mean(span=4).alias("rain_ewm_1h"), 
        pl.col("rain_step").ewm_mean(span=12).alias("rain_ewm_3h"),
        pl.col("rain_step").ewm_mean(span=24).alias("rain_ewm_6h"), 
        pl.col("rain_step").ewm_mean(span=48).alias("rain_ewm_12h"),
        pl.col("rain_step").ewm_mean(span=96).alias("rain_ewm_24h"), 
        pl.col("rain_step").ewm_mean(span=288).alias("rain_ewm_72h"),
        pl.col("rain_step").cum_sum().alias("rain_cum"),
        pl.col("rain_step").diff().fill_null(0.0).alias("rain_acceleration"),
        pl.col("rain_step").rolling_max(12).fill_null(0.0).alias("rain_rolling_max_3h"),
        (pl.col("rain_step") == 0).cast(pl.Int32).cum_sum().alias("dry_steps_count"),
        pl.col("rain_step").rolling_sum(4).fill_null(0.0).alias("rain_roll_1h"),
    ])
    
    df_rain = df_rain.with_columns([
        pl.col("rain_acceleration").diff().fill_null(0.0).alias("rain_jerk"),
        (pl.col("rain_ewm_1h") - pl.col("rain_ewm_6h")).alias("rain_trend_fast"),
        (pl.col("rain_ewm_6h") - pl.col("rain_ewm_24h")).alias("rain_trend_slow"),
        (pl.col("rain_cum") - pl.col("rain_roll_1h")).alias("antecedent_rain"),
        (pl.col("rain_step") / (df_rain["rain_step"].max() + 0.001)).alias("rain_relative_intensity"),
        pl.col("rain_ewm_1h").shift(-4).fill_null(0.0).alias("rain_future_1h"),
        pl.col("rain_ewm_3h").shift(-12).fill_null(0.0).alias("rain_future_3h"),
    ])

    max_t = df_rain["timestep"].max()
    event_total = df_rain["rain_step"].sum()
    df_rain = df_rain.with_columns([
        (pl.col("timestep")/max_t).alias("time_phase"),
        np.sin(2*np.pi*pl.col("timestep")/max_t).alias("time_sin"),
        np.cos(2*np.pi*pl.col("timestep")/max_t).alias("time_cos"),
        (pl.col("rain_cum")/(event_total+0.001)).alias("rain_cum_pct"),
        pl.lit(event_total).alias("event_total_rain"),
        pl.lit(df_rain["rain_step"].max()).alias("event_max_rain")
    ])

    data_list = []
    for n_type, pattern in [(1, "*1d_nodes_dynamic*.csv"), (2, "*2d_nodes_dynamic*.csv")]:
        f_dyn = get_safe_files(f"{folder}/{pattern}")
        if not f_dyn: continue
        try: df_dyn = pl.read_csv(f_dyn[0]).with_columns(pl.col("timestep").cast(pl.Int32))
        except: continue
        if is_train and "water_level" not in df_dyn.columns: continue
        
        id_c = "node_idx" if "node_idx" in df_dyn.columns else "node_id"
        
        if "water_level" in df_dyn.columns:
            anchors = df_dyn.filter(pl.col("timestep") < 10).group_by(id_c).agg([
                pl.col("water_level").first().alias("init_wl_first"),
                pl.col("water_level").last().alias("init_wl_last"),
                pl.col("water_level").mean().alias("init_wl_mean"),
                pl.col("water_level").std().fill_null(0.0).alias("init_wl_std"),
                pl.col("water_level").max().alias("init_wl_max"),
                pl.col("water_level").min().alias("init_wl_min"),
                (pl.col("water_level").last() - pl.col("water_level").first()).alias("init_wl_trend"),
                ((pl.col("water_level").last() - pl.col("water_level").first()) / 9.0).alias("init_wl_velocity")
            ])
            df_dyn = df_dyn.join(anchors, on=id_c, how="left")
        else:
            for col in ["init_wl_first", "init_wl_last", "init_wl_mean", "init_wl_std", "init_wl_max", "init_wl_min", "init_wl_trend", "init_wl_velocity"]:
                df_dyn = df_dyn.with_columns(pl.lit(0.0).alias(col))
            
        df_merged = df_dyn.join(df_rain, on="timestep", how="left")
        
        explicit_cols = [
            "time_idx", "time_from_peak", "rain_ewm_1h", "rain_ewm_3h", "rain_ewm_6h", "rain_ewm_12h", "rain_ewm_24h", "rain_ewm_72h", 
            "rain_cum", "rain_acceleration", "rain_rolling_max_3h", "dry_steps_count", "rain_roll_1h", "rain_jerk", "rain_trend_fast", 
            "rain_trend_slow", "antecedent_rain", "rain_relative_intensity", "rain_future_1h", "rain_future_3h", "time_phase", 
            "time_sin", "time_cos", "rain_cum_pct", "event_total_rain", "event_max_rain",
            "init_wl_first", "init_wl_last", "init_wl_mean", "init_wl_std", "init_wl_max", "init_wl_min", "init_wl_trend", "init_wl_velocity"
        ]
        
        selector = [pl.col(id_c).cast(pl.Int32).alias("node_id"), pl.lit(n_type).cast(pl.Int32).alias("node_type"), pl.col("timestep").cast(pl.Int32).alias("timestep_int")]
        for c in explicit_cols:
            if c in df_merged.columns: selector.append(pl.col(c).fill_null(0.0))
            else: selector.append(pl.lit(0.0).alias(c))
            
        if "water_level" in df_merged.columns: selector.append(pl.col("water_level").alias("target"))
        else: selector.append(pl.col("timestep").cast(pl.Float32).alias("target"))
        
        data_list.append(df_merged.select(selector))
        
    return pl.concat(data_list) if data_list else None

# ==========================================
# 3. BUILD GRAND MATRIX
# ==========================================
print("🚀 Assembling FULL Data Matrix V184 (PURIFIED ENGINE)...")
all_train_folders, all_test_folders = [], []
for split, lst in [("train", all_train_folders), ("test", all_test_folders)]:
    for m_id in [1, 2]:
        base = find_model_path(m_id, split)
        if base:
            for f in get_safe_files(f"{base}/*event_*"):
                try: lst.append((m_id, int(os.path.basename(f).split("_")[-1]), f))
                except: pass

train_dfs, test_dfs = [], []
for m, e, f in tqdm(all_train_folders, desc="Extract Train"):
    df = extract_features_timeseries(f, m, e, is_train=True)
    if df is not None: train_dfs.append(df.with_columns([pl.lit(m).cast(pl.Int32).alias("model_id"), pl.lit(e).cast(pl.Int32).alias("event_id")]))
df_train = pl.concat(train_dfs)

for m, e, f in tqdm(all_test_folders, desc="Extract Test"):
    df = extract_features_timeseries(f, m, e, is_train=False)
    if df is not None: test_dfs.append(df.with_columns([pl.lit(m).cast(pl.Int32).alias("model_id"), pl.lit(e).cast(pl.Int32).alias("event_id")]))
df_test = pl.concat(test_dfs)

del train_dfs, test_dfs; gc.collect()

print("   🧹 Safe Downcasting Float64 to Float32...")
def downcast_float(df):
    fc = [c for c in df.columns if df[c].dtype == pl.Float64]
    if fc: return df.with_columns([pl.col(c).cast(pl.Float32) for c in fc])
    return df
df_train = downcast_float(df_train)
df_test = downcast_float(df_test)

# --- NODE STATS (CLEAN BASE) ---
print("   📊 Computing Node Stats...")
event_peaks = df_train.group_by(["model_id", "event_id", "node_type", "node_id"]).agg([pl.col("target").max().alias("event_max_wl")])
df_train = df_train.join(event_peaks, on=["model_id", "event_id", "node_type", "node_id"], how="left")
valid_peaks = df_train.filter((pl.col("target") == pl.col("event_max_wl")) & (pl.col("event_max_wl") > pl.col("init_wl_last") + 0.01))
delay_stats = valid_peaks.group_by(["model_id", "event_id", "node_type", "node_id"]).agg([pl.col("time_from_peak").mean().alias("event_peak_delay")]).group_by(["model_id", "node_type", "node_id"]).agg([pl.col("event_peak_delay").mean().alias("node_avg_delay"), pl.col("event_peak_delay").std().fill_null(0.0).alias("node_std_delay")])

df_train = df_train.drop("event_max_wl")

stats = df_train.group_by(["model_id", "node_type", "node_id"]).agg([
    pl.col("target").mean().alias("node_mean"), pl.col("target").std().fill_null(0.0).alias("node_std"),
    pl.col("target").max().fill_null(0.0).alias("node_max"), pl.col("target").min().fill_null(0.0).alias("node_min"),
    pl.col("target").quantile(0.95).alias("node_p95")
]).with_columns((pl.col("node_max") - pl.col("node_min")).alias("node_range"))

# --- SAFE SIGNATURES (No Interaction Cannibalization) ---
print("   🧬 Computing Safe Signatures...")
node_event = df_train.group_by(["model_id", "node_type", "node_id", "event_id"]).agg([
    pl.col("target").max().alias("ev_max"), pl.col("target").min().alias("ev_min"), pl.col("target").first().alias("ev_start"),
    pl.col("target").std().alias("ev_std"), pl.col("target").mean().alias("ev_mean"), pl.col("rain_cum").max().alias("ev_rain_total")
]).with_columns([
    (pl.col("ev_max") - pl.col("ev_start")).alias("amplitude"),
    ((pl.col("ev_max") - pl.col("ev_min")) / (pl.col("ev_rain_total") + 0.001)).alias("sensitivity"),
    (pl.col("ev_std") / (pl.col("ev_mean").abs() + 0.001)).alias("flashiness")
])
signatures = node_event.group_by(["model_id", "node_type", "node_id"]).agg([
    pl.col("sensitivity").median().alias("sig_sensitivity"), pl.col("sensitivity").std().fill_null(0.0).alias("sig_sensitivity_var"),
    pl.col("flashiness").median().alias("sig_flashiness"), pl.col("amplitude").max().alias("sig_amplitude_max")
])
del node_event; gc.collect()

# --- 1-HOP NEIGHBOR SIGNATURES ---
print("   📡 Computing Neighbor Messaging...")
if df_undirected_graph is not None:
    edges_with_sigs = df_undirected_graph.join(signatures, left_on=["model_id", "node_type", "to_node"], right_on=["model_id", "node_type", "node_id"], how="inner")
    neighbor_sigs = edges_with_sigs.group_by(["model_id", "node_type", "from_node"]).agg([
        pl.col("sig_sensitivity").mean().alias("neighbor_sens_mean"), pl.col("sig_amplitude_max").max().alias("neighbor_amp_max")
    ]).rename({"from_node": "node_id"})
else:
    neighbor_sigs = pl.DataFrame({"model_id": pl.Series(dtype=pl.Int32), "node_type": pl.Series(dtype=pl.Int32), "node_id": pl.Series(dtype=pl.Int32), "neighbor_sens_mean": pl.Series(dtype=pl.Float64), "neighbor_amp_max": pl.Series(dtype=pl.Float64)})

print("   💾 Saving Node Stats & Signatures for Inference...")
stats.write_csv("models/v184_stats.csv")
signatures.write_csv("models/v184_signatures.csv")
delay_stats.write_csv("models/v184_delay_stats.csv")

# ==========================================
# 🛑 EXPLICIT SAFE JOINS
# ==========================================
print("   🔗 Executing Explicit Safe Joins...")
df_train = df_train.join(stats, on=["model_id", "node_type", "node_id"], how="left")
df_train = df_train.join(signatures, on=["model_id", "node_type", "node_id"], how="left")
df_train = df_train.join(neighbor_sigs, on=["model_id", "node_type", "node_id"], how="left")
df_train = df_train.join(df_static, on=["model_id", "node_type", "node_id"], how="left")
df_train = df_train.join(delay_stats, on=["model_id", "node_type", "node_id"], how="left")
if df_edges is not None: df_train = df_train.join(df_edges, on=["model_id", "node_type", "node_id"], how="left")
if df_cross is not None: df_train = df_train.join(df_cross, on=["model_id", "node_type", "node_id"], how="left")

df_test = df_test.join(stats, on=["model_id", "node_type", "node_id"], how="left")
df_test = df_test.join(signatures, on=["model_id", "node_type", "node_id"], how="left")
df_test = df_test.join(neighbor_sigs, on=["model_id", "node_type", "node_id"], how="left")
df_test = df_test.join(df_static, on=["model_id", "node_type", "node_id"], how="left")
df_test = df_test.join(delay_stats, on=["model_id", "node_type", "node_id"], how="left")
if df_edges is not None: df_test = df_test.join(df_edges, on=["model_id", "node_type", "node_id"], how="left")
if df_cross is not None: df_test = df_test.join(df_cross, on=["model_id", "node_type", "node_id"], how="left")

# ==========================================
# 🛑 BULLETPROOF INIT FALLBACKS
# ==========================================
print("   🛡️ Applying Bulletproof Init Fallbacks...")
def apply_fallbacks(df):
    wl_init_cols = ["init_wl_first", "init_wl_last", "init_wl_mean", "init_wl_max", "init_wl_min"]
    for col in wl_init_cols:
        if col in df.columns:
            df = df.with_columns(pl.when(pl.col(col).is_null() | (pl.col(col).abs() < 1e-6)).then(pl.col("node_mean")).otherwise(pl.col(col)).alias(col))
            
    zero_init_cols = ["init_wl_std", "init_wl_trend", "init_wl_velocity"]
    for col in zero_init_cols:
        if col in df.columns:
            df = df.with_columns(pl.col(col).fill_null(0.0))
            
    for c in df.columns:
        if c not in ["target", "event_id", "timestep_int", "model_id", "node_type", "node_id"]:
            df = df.with_columns(pl.col(c).fill_null(0.0))
            
    return df

df_train = apply_fallbacks(df_train)
df_test = apply_fallbacks(df_test)

print("   🎯 Applying Target Scaling & Dynamic Features...")
group_stds = df_train.group_by(["model_id", "node_type"]).agg([pl.col("target").std().alias("group_std")])
df_train = df_train.join(group_stds, on=["model_id", "node_type"], how="left")
df_train = df_train.with_columns([(pl.col("target") - pl.col("node_mean")).alias("residual"), (1.0 / (pl.col("group_std").pow(2) + 1e-6)).alias("sample_weight")])

def apply_dynamic_features(df):
    exprs = [
        (pl.col("time_from_peak") - pl.col("node_avg_delay")).alias("relative_time_to_peak"),
        (pl.col("rain_ewm_3h") * pl.col("node_avg_delay")).alias("interact_rain_delay"),
        (pl.col("rain_cum") / (pl.col("node_avg_delay").abs() + 1.0)).alias("interact_cum_delay"),
        (pl.col("rain_cum") / (pl.col("calc_depth") * pl.col("area") + 0.1)).alias("proxy_fill_ratio"),
        (pl.col("init_wl_last") - pl.col("node_mean")).alias("interact_anchor_dev"),
        (pl.col("rain_rolling_max_3h") * pl.col("node_range")).alias("interact_peak_range"),
        
        # 🟢 THE APPROVED GREENLIST:
        (pl.col("rain_cum") * pl.col("sig_sensitivity")).alias("sig_expected_wl_delta"),
        (pl.col("sig_amplitude_max") - (pl.col("init_wl_last") - pl.col("node_mean"))).alias("sig_headroom"),
        (pl.col("rain_ewm_6h") * pl.col("sig_sensitivity")).alias("sig_impact_6h"),
        (pl.col("rain_cum") * pl.col("neighbor_sens_mean")).alias("neighbor_expected_inflow")
    ]
    if "flow_accumulation" in df.columns: exprs.append((pl.col("rain_cum") * pl.col("flow_accumulation")).alias("interact_rain_accumulation"))
    if "curvature" in df.columns: exprs.append((pl.col("curvature") * pl.col("rain_ewm_3h")).alias("interact_curvature_rain"))
    if "total_capacity_proxy" in df.columns: exprs.append((pl.col("rain_ewm_1h") / (pl.col("total_capacity_proxy") + 0.001)).alias("capacity_utilization"))
    if "pipe_dia_max" in df.columns: exprs.append((pl.col("rain_cum") / (pl.col("pipe_dia_max").pow(2) * 3.14 / 4 + 0.01)).alias("surcharge_proxy"))
    if "coupled_1d_capacity" in df.columns: exprs.append((pl.col("rain_ewm_1h") / (pl.col("coupled_1d_capacity") + 0.1)).alias("cross_choke_risk_1h"))
    if "coupled_2d_flow_acc" in df.columns: exprs.append((pl.col("rain_cum") * pl.col("coupled_2d_flow_acc")).alias("cross_inlet_load"))
    return df.with_columns(exprs)

df_train = apply_dynamic_features(df_train)
df_test = apply_dynamic_features(df_test)

# 🔥 OOM FIX: PARQUET SWAP 🔥 
print("   💾 Swapping df_test to disk to save memory...")
df_test.write_parquet("test_swap_v184.parquet")
del df_test; gc.collect()

exclude = ["target", "residual", "sample_weight", "group_std", "model_id", "node_type", "node_id", "event_id", "timestep_int"]
all_feats = [c for c in df_train.columns if c not in exclude]

print(f"\n   ⚔️  Training 4 Spatial Models with MULTI-SEED BAGGING ({len(all_feats)} clean features)...")
models_dict = {}

for m_id in [1, 2]:
    for n_type in [1, 2]:
        name = f"m{m_id}_t{n_type}"
        subset = df_train.filter((pl.col("model_id") == m_id) & (pl.col("node_type") == n_type))
        if subset.height == 0: continue
            
        valid_feats = [f for f in all_feats if subset[f].abs().sum() > 0]
        print(f"   --- {name.upper()} ({len(valid_feats)} valid features) ---")
        
        X = subset.select(valid_feats).to_numpy().astype(np.float32)
        y = subset["residual"].to_numpy().astype(np.float32)
        w = subset["sample_weight"].to_numpy().astype(np.float32)
        
        del subset; gc.collect()
        
        dtrain = lgb.Dataset(X, label=y, weight=w, free_raw_data=True).construct()
        del X, y, w; gc.collect()
        
        params = CONFIG[f"params_{name}"]
        model_ensemble = []
        for seed in CONFIG["seeds"]:
            params['seed'] = seed
            model = lgb.train(params, dtrain, num_boost_round=CONFIG["rounds"])
            model_ensemble.append(model)
            # 💾
            model_path = f"models/lgbm_v184_m{m_id}_t{n_type}_seed{seed}.txt"
            model.save_model(model_path)
            
        # 💾
        with open(f"models/features_v184_m{m_id}_t{n_type}.json", "w") as f:
            json.dump(valid_feats, f)
            
        models_dict[(m_id, n_type)] = (model_ensemble, valid_feats)
        del dtrain; gc.collect()

del df_train; gc.collect()

# ==========================================
# 5. INFERENCE
# ==========================================
print("\n   🔮 Fast Inference...")
df_test = pl.read_parquet("test_swap_v184.parquet")

sample_path = get_safe_files(f"{CONFIG['raw_data_dir']}/**/sample_submission.csv")[0]
df_sub_orig = pl.read_csv(sample_path)
test_tasks = df_sub_orig.select(["model_id", "event_id"]).unique().to_dicts()

mapper_chunks = []
for task in tqdm(test_tasks, desc="Test Prep"):
    m, e = task['model_id'], task['event_id']
    sub_slice = df_sub_orig.filter((pl.col("model_id")==m) & (pl.col("event_id")==e))
    base = find_model_path(m, "test")
    if not base: continue
    folder = get_safe_files(f"{base}/event_{e}")[0]
    for n_type, pattern in [(1, "*1d_nodes_dynamic*.csv"), (2, "*2d_nodes_dynamic*.csv")]:
        files = get_safe_files(f"{folder}/{pattern}")
        if not files: continue
        df_dyn = pl.read_csv(files[0]).with_columns(pl.col("timestep").cast(pl.Int32))
        id_col = "node_idx" if "node_idx" in df_dyn.columns else "node_id"
        targets = df_dyn.filter(pl.col("water_level").is_null()).select([pl.col("timestep").cast(pl.Int32).alias("timestep_int"), pl.col(id_col).cast(pl.Int32).alias("node_id")])
        if targets.height > 0:
            targets = targets.sort(["node_id", "timestep_int"]).with_columns(pl.int_range(0, pl.len()).over("node_id").alias("seq_idx"))
            sub_nodes = sub_slice.filter(pl.col("node_type") == n_type).sort(["node_id", "row_id"]).with_columns(pl.int_range(0, pl.len()).over("node_id").alias("seq_idx"))
            mapped = sub_nodes.join(targets, on=["node_id", "seq_idx"], how="inner")
            mapper_chunks.append(mapped.select(["row_id", "model_id", "event_id", "node_type", "node_id", "timestep_int"]))

results = []
threads = os.cpu_count() or -1

for m_id in [1, 2]:
    for n_type in [1, 2]:
        subset = df_test.filter((pl.col("model_id") == m_id) & (pl.col("node_type") == n_type))
        if subset.height > 0 and (m_id, n_type) in models_dict:
            model_ensemble, features = models_dict[(m_id, n_type)]
            X_numpy = subset.select(features).to_numpy().astype(np.float32)
            preds = np.zeros(X_numpy.shape[0], dtype=np.float32)
            for model in model_ensemble:
                preds += model.predict(X_numpy, num_threads=threads)
            preds /= len(model_ensemble)
            results.append(subset.select(["model_id", "event_id", "node_type", "node_id", "timestep_int", "node_mean", "node_min"]).with_columns(pl.Series(preds).alias("pred_residual")))

df_res = pl.concat(results).with_columns(pl.max_horizontal(pl.col("node_mean") + pl.col("pred_residual"), pl.col("node_min")).alias("pred_water_level"))
mapper_df = pl.concat(mapper_chunks)
final_updates = mapper_df.join(df_res.select(["model_id", "event_id", "node_type", "node_id", "timestep_int", "pred_water_level"]), on=["model_id", "event_id", "node_type", "node_id", "timestep_int"], how="left")
df_final = df_sub_orig.join(final_updates.select(["row_id", "pred_water_level"]), on="row_id", how="left").join(stats, on=["model_id", "node_type", "node_id"], how="left")

global_mean = stats["node_mean"].mean()
out = df_final.with_columns(pl.col("pred_water_level").fill_null(pl.col("node_mean").fill_null(global_mean))).select([
    "row_id", "model_id", "event_id", "node_type", "node_id", pl.col("pred_water_level").alias("water_level")
]).sort("row_id")

out.write_csv(CONFIG['output_file'])
if os.path.exists("test_swap_v184.parquet"): os.remove("test_swap_v184.parquet")
print(f"✅ V184 THE PURIFIED ENGINE Saved: {CONFIG['output_file']}")

🏗️ Loading Static Physics & Undirected Graphs...
🚀 Assembling FULL Data Matrix V184 (PURIFIED ENGINE)...


Extract Test: 100%|██████████| 59/59 [00:22<00:00,  2.66it/s]


   🧹 Safe Downcasting Float64 to Float32...
   📊 Computing Node Stats...
   🧬 Computing Safe Signatures...
   📡 Computing Neighbor Messaging...
   💾 Saving Node Stats & Signatures for Inference...
   🔗 Executing Explicit Safe Joins...
   🛡️ Applying Bulletproof Init Fallbacks...
   🎯 Applying Target Scaling & Dynamic Features...
   💾 Swapping df_test to disk to save memory...

   ⚔️  Training 4 Spatial Models with MULTI-SEED BAGGING (98 clean features)...
   --- M1_T1 (72 valid features) ---
   --- M1_T2 (90 valid features) ---
   --- M2_T1 (72 valid features) ---
   --- M2_T2 (90 valid features) ---

   🔮 Fast Inference...


Test Prep: 100%|██████████| 59/59 [00:28<00:00,  2.04it/s]


✅ V184 THE PURIFIED ENGINE Saved: submission_v184_PURIFIED_ENGINE.csv


In [4]:
import pandas as pd
from scipy.signal import savgol_filter
from tqdm import tqdm

tqdm.pandas()

print("🌊 Creating PERFECT BLEND + PHYSICAL SMOOTHING (Savgol)...")

# 1. Load our best submissions
sub1 = pd.read_csv('submission_v182_ULTIMATE_SIGNATURES.csv') 
sub2 = pd.read_csv('submission_v184_PURIFIED_ENGINE.csv') 

assert (sub1['row_id'] == sub2['row_id']).all(), "🚨 Error: row_id mismatch!"

# 2. Pure 50/50 blend
df_blend = sub1.copy()
df_blend['water_level'] = (sub1['water_level'] + sub2['water_level']) / 2.0

# 3. Savitzky-Golay Filter
print("📈 Applying Savitzky-Golay filter to each time series...")

def apply_savgol(group):
    if len(group) < 5:
        return group
    
    group = group.sort_values('row_id')
    
    # window_length=5, polyorder=2 (strictly preserving the peaks)
    group['water_level'] = savgol_filter(group['water_level'], window_length=5, polyorder=2)
    return group

# Group by node and apply the filter
df_smoothed = df_blend.groupby(['model_id', 'event_id', 'node_type', 'node_id'], group_keys=False).progress_apply(apply_savgol)

output_name = 'submission_PURE_BLEND_SAVGOL_FIXED.csv'
df_smoothed.sort_values('row_id').to_csv(output_name, index=False)

print(f"✅ Done! File saved as: {output_name}")

🌊 Creating PERFECT BLEND + PHYSICAL SMOOTHING (Savgol)...
📈 Applying Savitzky-Golay filter to each time series...


100%|██████████| 243167/243167 [02:24<00:00, 1687.28it/s]


✅ Done! File saved as: submission_PURE_BLEND_SAVGOL_FIXED.csv
